<a href="https://colab.research.google.com/github/nikitask14/neural-networks-pytorch-from-first-principles/blob/main/03_validation_and_classification/012_dataset_dataloader_and_minibatches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
import torch
import torch.nn as nn

In [82]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)


  def forward(self, x):
    x = self.layer1(x)
    x = torch.relu(x)

    x = self.layer2(x)
    return x


x_train = torch.tensor([
    [1,2],
    [3,4],
    [5,6],
    [7,8],
    [9,10],
    [11,12]
], dtype = torch.float32)


# Class 1 if the sum of the two features is > 10
# Otherwise class 0
y_train = torch.tensor([
    [0],
    [0],
    [1],
    [1],
    [1],
    [1]
], dtype = torch.float32)

print(x_train.shape)
print(y_train.shape)

torch.Size([6, 2])
torch.Size([6, 1])


In [83]:
from torch.utils.data import TensorDataset


In [84]:
train_dataset = TensorDataset(x_train, y_train)

This creates a dataset where corresponding entries are paired:

$$ (x\_train[0],y\_train[0]) $$ $$ (x\_train[1],y\_train[1]) $$

and so on.

In [85]:
print(len(train_dataset))
print(train_dataset[0])

6
(tensor([1., 2.]), tensor([0.]))


The next question we want to answer is:

####How do we automatically take those 6 examples and give them to the model in smaller groups, instead of manually slicing the tensors ourselves?

In [86]:
from torch.utils.data import DataLoader

In [87]:
# shuffle=True means:
# before forming batches for a training pass, randomly reorder the example
train_loader = DataLoader(train_dataset, batch_size = 2, shuffle = True)
print(len(train_loader))

3


In [88]:
#“For every batch from the loader, separate the inputs and labels
# and call them x_batch and y_batch.”
for x_batch, y_batch in train_loader:
  print("x_batch:", x_batch)
  print("x_batch shape", x_batch.shape)
  print("y_batch:", y_batch)
  print("y_batch Shape:", y_batch.shape)

  print("--------------------------")
  print()
  print()

x_batch: tensor([[7., 8.],
        [1., 2.]])
x_batch shape torch.Size([2, 2])
y_batch: tensor([[1.],
        [0.]])
y_batch Shape: torch.Size([2, 1])
--------------------------


x_batch: tensor([[ 9., 10.],
        [ 3.,  4.]])
x_batch shape torch.Size([2, 2])
y_batch: tensor([[1.],
        [0.]])
y_batch Shape: torch.Size([2, 1])
--------------------------


x_batch: tensor([[ 5.,  6.],
        [11., 12.]])
x_batch shape torch.Size([2, 2])
y_batch: tensor([[1.],
        [1.]])
y_batch Shape: torch.Size([2, 1])
--------------------------




In [89]:
# use mini-batches inside the training loop.

model = MyModel()
loss_fn = nn.BCEWithLogitsLoss()
optimiser = torch.optim.SGD(model.parameters(), lr = 0.01)

model.train()

for epoch in range(100):
  for x_batch, y_batch in train_loader:
    optimiser.zero_grad()
    train_logits = model(x_batch)
    train_loss = loss_fn(train_logits, y_batch)
    train_loss.backward()
    optimiser.step()

